In [12]:
!pip install torch datasets

In [13]:
from datasets import load_dataset

dataset = load_dataset("squad")

train_data = dataset["train"].select(range(500))  # kiçik hissə

In [14]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

--2026-04-05 11:50:52--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2026-04-05 11:50:52--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-04-05 11:50:53--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [15]:
import numpy as np

def load_glove(file_path):
    embeddings = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            embeddings[word] = vector
    return embeddings

glove = load_glove("glove.6B.100d.txt")

In [16]:
def build_vocab(data):
    vocab = set()
    for sample in data:
        vocab.update(sample["context"].lower().split())
        vocab.update(sample["question"].lower().split())
    return list(vocab)

vocab = build_vocab(train_data)
word2idx = {w: i+1 for i, w in enumerate(vocab)}

In [17]:
embedding_dim = 100
embedding_matrix = np.zeros((len(word2idx)+1, embedding_dim))

for word, i in word2idx.items():
    if word in glove:
        embedding_matrix[i] = glove[word]
    else:
        embedding_matrix[i] = np.random.normal(scale=0.6, size=(embedding_dim,))

In [18]:
def encode(text):
    return [word2idx.get(w, 0) for w in text.lower().split()]

In [19]:
import torch
import torch.nn as nn

class SimpleBiDAF(nn.Module):
    def __init__(self, embedding_matrix):
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32),
            freeze=False
        )

        self.linear_start = nn.Linear(200, 1)
        self.linear_end = nn.Linear(200, 1)

    def forward(self, context_ids, question_ids):
        context_emb = self.embedding(context_ids)
        question_emb = self.embedding(question_ids)

        similarity = torch.matmul(context_emb, question_emb.transpose(1,2))
        attention = torch.softmax(similarity, dim=-1)
        attended_question = torch.matmul(attention, question_emb)

        combined = torch.cat([context_emb, attended_question], dim=-1)

        start_logits = self.linear_start(combined).squeeze(-1)
        end_logits = self.linear_end(combined).squeeze(-1)

        return start_logits, end_logits

In [20]:
model = SimpleBiDAF(embedding_matrix)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

In [29]:
for sample in train_data.select(range(200)):
    context_ids = torch.tensor([encode(sample["context"])])
    question_ids = torch.tensor([encode(sample["question"])])

    context = sample["context"]
    answer_start_char = sample["answers"]["answer_start"][0]
    answer_text = sample["answers"]["text"][0]

    context_words = context.split()

    char_count = 0
    start_idx = 0

    for i, word in enumerate(context_words):
        if char_count >= answer_start_char:
            start_idx = i
            break
        char_count += len(word) + 1

    end_idx = start_idx + len(answer_text.split())


    max_len = context_ids.shape[1]
    start_idx = min(start_idx, max_len - 1)
    end_idx = min(end_idx, max_len - 1)

    start_pos = torch.tensor([start_idx])
    end_pos = torch.tensor([end_idx])

    start_logits, end_logits = model(context_ids, question_ids)

    loss_start = loss_fn(start_logits, start_pos)
    loss_end = loss_fn(end_logits, end_pos)

    loss = (loss_start + loss_end) / 2

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [30]:
def predict(question, context):
    context_ids = torch.tensor([encode(context)])
    question_ids = torch.tensor([encode(question)])

    start_logits, end_logits = model(context_ids, question_ids)

    start = torch.argmax(start_logits)
    end = torch.argmax(end_logits)

    if end < start:
        end = start + 1

    words = context.split()
    answer = " ".join(words[start:end])

    return start.item(), end.item(), answer

In [31]:
start, end, answer = predict(
    "What is the capital of Azerbaijan?",
    "Baku is the capital of Azerbaijan."
)

print("Start:", start)
print("End:", end)
print("Answer:", answer)

Start: 0
End: 1
Answer: Baku
